In [4]:
# Jupyter Notebook Code for Symmetric GPT Data Analysis
# Run each cell individually in your Jupyter notebook

# Cell 1: Import libraries and load data
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Load the data
csv_path = 'sym_GPT_data.csv'  # Update this path as needed
df = pd.read_csv(csv_path)

print(f"Loaded {len(df)} records")
print(f"Columns: {list(df.columns)}")
print(f"\nUnique model combinations: {df['model_combination'].nunique()}")
print(f"Unique recipes: {df['recipe'].nunique()}")

# Check what's in the analysis column
if 'analysis' in df.columns:
    print(f"Unique analysis values: {df['analysis'].unique()}")
    analysis_counts = df['analysis'].value_counts()
    print(f"Analysis distribution:\n{analysis_counts}")

# Cell 2: Basic data overview
print("Model combinations:")
print(df['model_combination'].value_counts())
print("\nRecipes:")
print(df['recipe'].value_counts())

if 'analysis' in df.columns:
    print("\nAnalysis types:")
    print(df['analysis'].value_counts())

print("\nAgent distribution:")
print(df['agent_id'].value_counts())

# Cell 3: Table 1 - Model combination aggregated across all recipes
print("="*80)
print("TABLE 1: Model Combination Aggregated Across All Recipes")
print("="*80)

# Basic aggregations
model_overall = df.groupby('model_combination').agg({
    'sample_index': [
        ('total_samples', 'count'),
        ('unique_samples', 'nunique')
    ],
    'agent_id': [
        ('unique_agents', 'nunique')
    ],
    'recipe': [
        ('unique_recipes', 'nunique')
    ],
    'experiment_file': [
        ('unique_experiments', 'nunique')
    ]
}).round(3)

# Flatten column names
model_overall.columns = [f"{col[1]}_{col[0]}" if col[1] != col[0] else col[0] for col in model_overall.columns]

# Add analysis column aggregations if it exists
if 'analysis' in df.columns:
    analysis_agg = df.groupby('model_combination')['analysis'].agg([
        ('most_common_analysis', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A'),
        ('analysis_diversity', 'nunique')
    ])
    
    # Add individual analysis type counts
    for analysis_type in df['analysis'].unique():
        if pd.notna(analysis_type):  # Skip NaN values
            analysis_agg[f'{analysis_type}_count'] = df.groupby('model_combination')['analysis'].apply(
                lambda x: (x == analysis_type).sum()
            )
            analysis_agg[f'{analysis_type}_pct'] = df.groupby('model_combination')['analysis'].apply(
                lambda x: (x == analysis_type).sum() / len(x) * 100
            ).round(2)
    
    # Combine with basic aggregations
    model_overall = pd.concat([model_overall, analysis_agg], axis=1)

display(HTML("<h3>Model Performance Across All Recipes</h3>"))
display(model_overall)

# Cell 4: Table 2 - Model combination and recipe-specific aggregation
print("="*80)
print("TABLE 2: Model Combination and Recipe-Specific Aggregation")
print("="*80)

model_recipe = df.groupby(['model_combination', 'recipe']).agg({
    'sample_index': [
        ('total_samples', 'count'),
        ('unique_samples', 'nunique')
    ],
    'agent_id': [
        ('unique_agents', 'nunique')
    ],
    'experiment_file': [
        ('unique_experiments', 'nunique')
    ]
}).round(3)

# Flatten column names
model_recipe.columns = [f"{col[1]}_{col[0]}" if col[1] != col[0] else col[0] for col in model_recipe.columns]

# Add analysis column aggregations for recipe-specific view
if 'analysis' in df.columns:
    analysis_recipe_agg = df.groupby(['model_combination', 'recipe'])['analysis'].agg([
        ('most_common_analysis', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A'),
        ('analysis_diversity', 'nunique')
    ])
    
    # Add individual analysis type counts for recipe view
    for analysis_type in df['analysis'].unique():
        if pd.notna(analysis_type):  # Skip NaN values
            analysis_recipe_agg[f'{analysis_type}_count'] = df.groupby(['model_combination', 'recipe'])['analysis'].apply(
                lambda x: (x == analysis_type).sum()
            )
            analysis_recipe_agg[f'{analysis_type}_pct'] = df.groupby(['model_combination', 'recipe'])['analysis'].apply(
                lambda x: (x == analysis_type).sum() / len(x) * 100
            ).round(2)
    
    # Combine with basic aggregations
    model_recipe = pd.concat([model_recipe, analysis_recipe_agg], axis=1)

display(HTML("<h3>Model Performance by Recipe</h3>"))
display(model_recipe)

# Cell 5: Text content analysis (for 'say' and 'plan' columns)
print("="*80)
print("TEXT CONTENT ANALYSIS")
print("="*80)

# Analyze 'say' and 'plan' columns if they exist
text_analysis = {}

for col in ['say', 'plan']:
    if col in df.columns:
        # Basic text statistics
        df[f'{col}_length'] = df[col].astype(str).str.len()
        df[f'{col}_word_count'] = df[col].astype(str).str.split().str.len()
        
        text_stats = df.groupby('model_combination').agg({
            f'{col}_length': ['mean', 'std', 'min', 'max'],
            f'{col}_word_count': ['mean', 'std', 'min', 'max']
        }).round(2)
        
        text_stats.columns = [f"{col}_{stat}_{metric}" for stat, metric in text_stats.columns]
        text_analysis[col] = text_stats

# Display text analysis
for col, stats in text_analysis.items():
    display(HTML(f"<h4>{col.capitalize()} Text Statistics by Model</h4>"))
    display(stats)

# Cell 6: Summary statistics and insights
print("="*80)
print("SUMMARY STATISTICS")
print("="*80)

print(f"Total unique model combinations: {df['model_combination'].nunique()}")
print(f"Total unique recipes: {df['recipe'].nunique()}")
print(f"Total samples: {len(df)}")
print(f"Unique agents: {df['agent_id'].nunique()}")
print(f"Unique experiments: {df['experiment_file'].nunique()}")

if 'analysis' in df.columns:
    analysis_summary = df['analysis'].value_counts()
    print(f"\nOverall analysis distribution:")
    for analysis, count in analysis_summary.items():
        percentage = (count / len(df)) * 100
        print(f"  {analysis}: {count} ({percentage:.2f}%)")

# Agent distribution
agent_summary = df['agent_id'].value_counts()
print(f"\nAgent distribution:")
for agent, count in agent_summary.items():
    percentage = (count / len(df)) * 100
    print(f"  Agent {agent}: {count} ({percentage:.2f}%)")

# Store the dataframes for further analysis
print("\nDataFrames stored as:")
print("- model_overall: Model performance across all recipes")
print("- model_recipe: Model performance by recipe")
print("- text_analysis: Text statistics (if say/plan columns exist)")

# Cell 7: Optional - Create simple visualizations
# Uncomment and run if you want basic plots

# import matplotlib.pyplot as plt
# import seaborn as sns

# # Plot 1: Analysis distribution by model combination
# if 'analysis' in df.columns:
#     plt.figure(figsize=(12, 6))
#     analysis_crosstab = pd.crosstab(df['model_combination'], df['analysis'])
#     analysis_crosstab.plot(kind='bar', stacked=True)
#     plt.title('Analysis Distribution by Model Combination')
#     plt.xlabel('Model Combination')
#     plt.ylabel('Count')
#     plt.xticks(rotation=45, ha='right')
#     plt.tight_layout()
#     plt.show()

# # Plot 2: Sample distribution by recipe
# plt.figure(figsize=(10, 6))
# recipe_counts = df.groupby(['recipe', 'model_combination']).size().unstack(fill_value=0)
# recipe_counts.plot(kind='bar')
# plt.title('Sample Distribution by Recipe and Model Combination')
# plt.xlabel('Recipe')
# plt.ylabel('Count')
# plt.xticks(rotation=45, ha='right')
# plt.tight_layout()
# plt.show()


Loaded 17049 records
Columns: ['timestamp', 'model_combination', 'recipe', 'experiment_file', 'sample_index', 'agent_id', 'game_timestamp', 'analysis', 'say', 'plan']

Unique model combinations: 25
Unique recipes: 4
Unique analysis values: ['The game has just started, and we need to complete the "Baked Pumpkin Slices" order. The recipe requires us to first cut a pumpkin into slices, then place the slices in the oven and bake for 3 timesteps, and finally serve the baked slices. Since both players are starting with nothing, we need to coordinate to avoid both trying to pick up the pumpkin at the same time. I will pick up the pumpkin first, cut it, and place the slices in the oven. Then, I will initiate the baking process. Once the baking is complete, I will retrieve the slices and serve them. If I need help with any step, I will request it from my teammate.'
 "The game has just started, and both players are holding nothing. The recipe for Baked Pumpkin Slices requires cutting a pumpkin i

/var/folders/zs/vqkr9f1s7k7957sf_gd0vw940000gn/T/ipykernel_27069/310162458.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  analysis_agg[f'{analysis_type}_count'] = df.groupby('model_combination')['analysis'].apply(
/var/folders/zs/vqkr9f1s7k7957sf_gd0vw940000gn/T/ipykernel_27069/310162458.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  analysis_agg[f'{analysis_type}_pct'] = df.groupby('model_combination')['analysis'].apply(
/var/folders/zs/vqkr9f1s7k7957sf_gd0vw940000gn/T/ipykernel_27069/310162458.py:72: PerformanceWar

KeyboardInterrupt: 